In [1]:
translation_names = ['WRJTx', 'WRJTy', 'WRJTz']
rot_names = ['WRJRx', 'WRJRy', 'WRJRz']
joint_names = [
    "thumb_cmc_roll",
    "thumb_cmc_yaw",
    "thumb_cmc_pitch",
    "thumb_mcp",
    "thumb_ip",
    "index_mcp_roll",
    "index_mcp_pitch",
    "index_pip",
    "index_dip",
    "middle_mcp_roll",
    "middle_mcp_pitch",
    "middle_pip",
    "middle_dip",
    "ring_mcp_pitch",
    "ring_pip",
    "ring_dip",
    "pinky_mcp_pitch",
    "pinky_pip",
    "pinky_dip"
]

thumb = {
    "thumb_cmc_roll",
    "thumb_cmc_yaw",
    "thumb_cmc_pitch",
    "thumb_mcp",
    "thumb_ip"
}

In [2]:
import numpy as np
from collections import defaultdict
import scipy.spatial.transform as transform
import numpy as np
from scipy.spatial.transform import Rotation as R

grasp_poses =np.load('/home/guizhewei/guizhewei/Dexycb_dataset/grasp_poses_1030_1518_new.npy', allow_pickle=True).item()
# grasp_poses =np.load('/home/ubuntu/Documents/DexYCB/grasp_poses_opt.npy', allow_pickle=True).item()
grasp_poses.keys()
# obj_idx = 3

dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49])

In [3]:
# grasp_poses[obj_idx].keys()


In [4]:
# grasp_poses[obj_idx]['target_object_name']

In [5]:
# grasp_poses[obj_idx]['robot_pose'][0]

In [6]:
import numpy as np
from scipy.spatial.transform import Rotation as R

def euler_to_rotation_matrix(euler_angles):
    rotation = R.from_euler('XYZ', euler_angles, degrees=False)
    return rotation.as_matrix()


def quaternion_to_rotation_matrix(quaternion):
    rotation = R.from_quat(quaternion)
    return rotation.as_matrix()


def object_pose_to_matrix(position, quaternion):
    """
    Converts object pose (position and quaternion) to a 4x4 transformation matrix.
    
    Parameters:
    - position: (3,) numpy.ndarray, position [x, y, z]
    - quaternion: (4,) numpy.ndarray, [x, y, z, w] quaternion
    
    Returns:
    - transformation_matrix: (4, 4) numpy.ndarray, corresponding transformation matrix
    """
    quaternion = np.concatenate([quaternion[1:4], quaternion[0:1]]) # wxyz-> xyzw
    rotation_matrix = quaternion_to_rotation_matrix(quaternion)
    transformation_matrix = np.eye(4)
    transformation_matrix[:3, :3] = rotation_matrix
    transformation_matrix[:3, 3] = position
    return transformation_matrix


def hand_pose_to_matrix(position, quaternion):
    """
    Converts object pose (position and quaternion) to a 4x4 transformation matrix.
    
    Parameters:
    - position: (3,) numpy.ndarray, position [x, y, z]
    - quaternion: (4,) numpy.ndarray, [x, y, z, w] quaternion
    
    Returns:
    - transformation_matrix: (4, 4) numpy.ndarray, corresponding transformation matrix
    """
    rotation_matrix = quaternion_to_rotation_matrix(quaternion)
    transformation_matrix = np.eye(4)
    transformation_matrix[:3, :3] = rotation_matrix
    transformation_matrix[:3, 3] = position
    return transformation_matrix

In [7]:
def get_obj_centric_pose_for_opt(grasp_poses, obj_idx):
    ''' Obj pose '''
    obj_pos = grasp_poses[obj_idx]['target_pose_world'][0].p
    obj_quat = grasp_poses[obj_idx]['target_pose_world'][0].q
    object_pose = object_pose_to_matrix(obj_pos, obj_quat)
    # print(f"Object Position: {obj_pos}, Object Quaternion: {obj_quat}")
    # print(f"Object Pose: {object_pose}")

    ''' Hand pose '''
    hand_pos = grasp_poses[obj_idx]['robot_pose'][0][:3]
    hand_euler = grasp_poses[obj_idx]['robot_pose'][0][3:6]
    hand_quat = transform.Rotation.from_euler('XYZ', hand_euler, degrees=False).as_quat()
    hand_6drot = transform.Rotation.from_euler('XYZ', hand_euler, degrees=False).as_matrix()
    hand_6drot = hand_6drot[:, :2].T.ravel().tolist()
    # print(f"Hand Position: {hand_pos}, Hand Quaternion: {hand_quat}")
    # print(f"Hand 6drot: {hand_6drot}")
    # print(grasp_poses[obj_idx]['robot_pose'][0])

    ''' object-centric '''
    W_T_O = object_pose
    W_T_H = hand_pose_to_matrix(hand_pos, hand_quat)

    O_T_H = np.linalg.inv(W_T_O) @ W_T_H
    print(O_T_H)
    t_oh  = O_T_H[:3, 3]
    R_oh  = O_T_H[:3, :3]

    euler_oh = R.from_matrix(R_oh).as_euler('XYZ', degrees=False)
    print(f"Object-Centric Hand Euler (XYZ, rad): {euler_oh}")
    print(f"Object-Centric Hand Position: {t_oh}")

    hand_6drot = R_oh
    hand_6drot = hand_6drot[:, :2].T.ravel().tolist()
    hand_pos = t_oh
    hand_euler = euler_oh

    return hand_pos, hand_euler, object_pose, hand_6drot

In [8]:
grasp_poses

{0: {'target_object_name': '024_bowl',
  'target_object_idx': 13,
  'target_pose_camera': [array([-0.5440993 , -0.3518824 ,  0.58101845,  0.49249598,  0.24959724,
          -0.14965104,  0.90708315], dtype=float32)],
  'target_pose_world': [Pose([0.506014, 0.254884, 0.18723], [-0.616193, -0.552497, -0.517738, -0.216795])],
  'camera_pose': Pose([1.01468, 0.777723, 0.799924], [0.0533614, -0.230272, -0.91078, 0.338536]),
  'robot_names': [<RobotName.panda: 8>],
  'robot_pose': [array([ 4.11953300e-01,  3.88929009e-01,  1.98116571e-01,  1.95376718e+00,
           4.00109410e-01,  4.82435703e-01,  3.78266871e-02, -2.61000007e-01,
          -2.52918899e-01,  1.13561533e-01,  1.26367614e-01, -6.76638305e-01,
           1.84040219e-02, -1.00000005e-03,  9.95480402e-02,  1.22513401e-01,
          -7.40991607e-02,  3.65101802e-03, -1.00000005e-03,  9.95480402e-02,
           1.22513401e-01, -2.31407154e-02,  4.07526632e-03, -1.14150005e-03,
          -1.85102583e-02])],
  'hand_type': <HandType

In [9]:
ycb_optimize_dataset_list = []

for obj_idx in grasp_poses.keys():
    hand_pos, hand_euler, object_pose, hand_6drot = get_obj_centric_pose_for_opt(grasp_poses, obj_idx)
    robot_pose_joint = grasp_poses[obj_idx]['robot_pose'][0]
    map_idx = [0, 5, 10, 15, 18, 1, 6, 11, 16, 2, 7, 12, 17, 3, 8, 13, 4, 9, 14]
    mapped_joint = [robot_pose_joint[6:][i] for i in map_idx]
    # print("mapped_joint", mapped_joint)
    robot_joint_dict = defaultdict(float)
    for i, joint_name in enumerate(joint_names):
        robot_joint_dict[joint_name] = mapped_joint[i]
    for i, name in enumerate(translation_names):
        robot_joint_dict[name] = hand_pos[i]
    for i, name in enumerate(rot_names):
        robot_joint_dict[name] = hand_euler[i]

    ycb_optimize_dataset = defaultdict(dict)
    ycb_optimize_dataset['qpos'] = robot_joint_dict
    print("obj_idx", obj_idx)
    print("ycb_optimize_dataset['qpos']", ycb_optimize_dataset['qpos'])
    ycb_optimize_dataset['object_code'] = grasp_poses[obj_idx]['target_object_name']
    ycb_optimize_dataset['object_pose'] = object_pose
    ycb_optimize_dataset['hand_rot6d'] = hand_6drot
    ycb_optimize_dataset['idx'] = obj_idx

    ycb_optimize_dataset_list.append(ycb_optimize_dataset)

ycb_optimize_dataset_list

[[ 0.20205298 -0.87709043 -0.43576023  0.07337005]
 [ 0.79848704  0.40516387 -0.44526474  0.02078446]
 [ 0.56709174 -0.25798183  0.7822099  -0.14532335]
 [ 0.          0.          0.          1.        ]]
Object-Centric Hand Euler (XYZ, rad): [ 0.5174943  -0.45088275  1.3443791 ]
Object-Centric Hand Position: [ 0.07337005  0.02078446 -0.14532335]
obj_idx 0
ycb_optimize_dataset['qpos'] defaultdict(<class 'float'>, {'thumb_cmc_roll': 0.03782668709754944, 'thumb_cmc_yaw': -0.6766383051872253, 'thumb_cmc_pitch': -0.07409916073083878, 'thumb_mcp': -0.023140715435147285, 'thumb_ip': -0.018510258276574314, 'index_mcp_roll': -0.26100000739097595, 'index_mcp_pitch': 0.018404021859169006, 'index_pip': 0.0036510180216282606, 'index_dip': 0.004075266315741465, 'middle_mcp_roll': -0.25291889905929565, 'middle_mcp_pitch': -0.0010000000474974513, 'middle_pip': -0.0010000000474974513, 'middle_dip': -0.0011415000542183407, 'ring_mcp_pitch': 0.11356153339147568, 'ring_pip': 0.09954804017096758, 'ring_di

[defaultdict(dict,
             {'qpos': defaultdict(float,
                          {'thumb_cmc_roll': 0.03782668709754944,
                           'thumb_cmc_yaw': -0.6766383051872253,
                           'thumb_cmc_pitch': -0.07409916073083878,
                           'thumb_mcp': -0.023140715435147285,
                           'thumb_ip': -0.018510258276574314,
                           'index_mcp_roll': -0.26100000739097595,
                           'index_mcp_pitch': 0.018404021859169006,
                           'index_pip': 0.0036510180216282606,
                           'index_dip': 0.004075266315741465,
                           'middle_mcp_roll': -0.25291889905929565,
                           'middle_mcp_pitch': -0.0010000000474974513,
                           'middle_pip': -0.0010000000474974513,
                           'middle_dip': -0.0011415000542183407,
                           'ring_mcp_pitch': 0.11356153339147568,
                     

In [10]:
ycb_optimize_dataset_list

[defaultdict(dict,
             {'qpos': defaultdict(float,
                          {'thumb_cmc_roll': 0.03782668709754944,
                           'thumb_cmc_yaw': -0.6766383051872253,
                           'thumb_cmc_pitch': -0.07409916073083878,
                           'thumb_mcp': -0.023140715435147285,
                           'thumb_ip': -0.018510258276574314,
                           'index_mcp_roll': -0.26100000739097595,
                           'index_mcp_pitch': 0.018404021859169006,
                           'index_pip': 0.0036510180216282606,
                           'index_dip': 0.004075266315741465,
                           'middle_mcp_roll': -0.25291889905929565,
                           'middle_mcp_pitch': -0.0010000000474974513,
                           'middle_pip': -0.0010000000474974513,
                           'middle_dip': -0.0011415000542183407,
                           'ring_mcp_pitch': 0.11356153339147568,
                     

In [11]:
# store dict in a specified path as npy file
import os
import json
output_path = '/home/guizhewei/guizhewei/grasp_pose_dataset/unoptimized/dexycb_1030_1518_omnihand.npy'
if not os.path.exists(os.path.dirname(output_path)):
    os.makedirs(os.path.dirname(output_path))
np.save(output_path, ycb_optimize_dataset_list, allow_pickle=True)